In [1]:
# ============================================================
# Cell 0 | 环境、项目根目录、路径
# ============================================================
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit import RDLogger

print("python =", sys.executable)
print("cwd    =", os.getcwd())

# 关闭 RDKit 刷屏日志
RDLogger.DisableLog("rdApp.error")
RDLogger.DisableLog("rdApp.warning")


def find_project_root(start: Path) -> Path:
    """
    自动向上寻找包含 data 和 notebooks 文件夹的原项目根目录。
    """
    start = start.resolve()

    for p in [start] + list(start.parents):
        if (
            (p / "data").exists()
            and (p / "notebooks").exists()
        ):
            return p

    raise FileNotFoundError(
        "Cannot find project root containing "
        "'data' and 'notebooks' folders."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# 本次10点留出验证的独立根目录
ROOT = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("ROOT         =", ROOT)

RAW_DIR = ROOT / "data" / "viscosity" / "raw"
OUT_DIR = ROOT / "data" / "viscosity" / "interim"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV = RAW_DIR / "viscosity.csv"
RAW_XLSX = RAW_DIR / "viscosity.xlsx"

print("RAW_DIR =", RAW_DIR)
print("OUT_DIR =", OUT_DIR)
print("RAW_CSV exists =", RAW_CSV.exists())

if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"找不到10点留出后的训练数据：\n{RAW_CSV}\n"
        "请先完成00_构建10点留出验证集.ipynb。"
    )

python = D:\conda\envs\rdkit-env\python.exe
cwd    = D:\jupyter\IL_ML_Project\notebooks\小论文_粘度预测_CO2筛选\01_10点留出验证\01_预处理
PROJECT_ROOT = D:\jupyter\IL_ML_Project
ROOT         = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
RAW_DIR = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw
OUT_DIR = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim
RAW_CSV exists = True


In [2]:
# ============================================================
# Cell 1 | 读取原始数据
# ============================================================
if RAW_CSV.exists():
    df_raw = pd.read_csv(RAW_CSV)
    print(f"Read CSV : {RAW_CSV}")
elif RAW_XLSX.exists():
    df_raw = pd.read_excel(RAW_XLSX)
    print(f"Read XLSX: {RAW_XLSX}")

    # 顺手保存一份 csv 快照
    df_raw.to_csv(
        RAW_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Snapshot CSV saved to: {RAW_CSV}")
else:
    raise FileNotFoundError(
        f"Neither found:\n"
        f"{RAW_CSV}\n"
        f"{RAW_XLSX}\n"
        "请先把原始粘度文件放到 "
        "data/viscosity/raw/ 下"
    )

# 清理列名
df_raw.columns = [
    str(c).strip()
    for c in df_raw.columns
]

print("raw shape =", df_raw.shape)
print("columns   =", df_raw.columns.tolist())

df_raw.head()

Read CSV : D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\raw\viscosity.csv
raw shape = (24942, 6)
columns   = ['离子液体全称', 'cation_smiles', 'anion_smiles', 'temperature, k', 'pressure, kpa', 'Viscosity, Pa•s']


,离子液体全称,cation_smiles,anion_smiles,"temperature, k","pressure, kpa","Viscosity, Pa•s"
0,"(((oxybis(ethane-2,1-diyl))bis(oxy))bis(ethane...",CC[S+](CC)CCOCCOCCOCC[S+](CC)CC,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,298.15,101.325,0.128 ± 0.013
1,"(((oxybis(ethane-2,1-diyl))bis(oxy))bis(ethane...",CC[S+](CC)CCOCCOCCOCC[S+](CC)CC,N#C[N-]C#N,298.15,101.325,0.143 ± 0.016
2,"((ethane-1,2-diylbis(oxy))bis(ethane-2,1-diyl)...",CC[S+](CC)CCOCCOCC[S+](CC)CC,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,298.15,101.325,0.236 ± 0.038
3,"((ethane-1,2-diylbis(oxy))bis(ethane-2,1-diyl)...",CC[S+](CC)CCOCCOCC[S+](CC)CC,N#C[N-]C#N,298.15,101.325,0.361 ± 0.075
4,"(1R,2S)-(-)-dimethylephedrinium bis(trifluorom...",C[C@H]([N+](C)(C)C)[C@H](O)c1ccccc1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,296.25,100.000,0.482 ± 0.039


In [3]:
# ============================================================
# Cell 2 | 自动识别关键列
# ============================================================
cols_lower = {
    c: str(c).lower()
    for c in df_raw.columns
}


def pick_col(candidates, must=True):
    hits = []

    for c in df_raw.columns:
        lc = cols_lower[c]

        if any(
            keyword in lc
            for keyword in candidates
        ):
            hits.append(c)

    if len(hits) == 0:
        if must:
            raise ValueError(
                f"找不到匹配列: {candidates}\n"
                f"当前列名: {df_raw.columns.tolist()}"
            )

        return None

    return hits[0]


# smiles列
cat_col = pick_col(
    [
        "cation_smiles",
        "cation smiles",
        "cation",
    ]
)

ani_col = pick_col(
    [
        "anion_smiles",
        "anion smiles",
        "anion",
    ]
)

# T / P / viscosity值列
t_col = pick_col(
    ["temperature"]
)

p_col = pick_col(
    ["pressure"]
)

vis_col = pick_col(
    [
        "viscosity",
        "eta",
    ]
)

# 单独的单位列（如果存在）
t_unit_col = pick_col(
    [
        "temperature_unit",
        "temp_unit",
        "t_unit",
    ],
    must=False,
)

p_unit_col = pick_col(
    [
        "pressure_unit",
        "press_unit",
        "p_unit",
    ],
    must=False,
)

vis_unit_col = pick_col(
    [
        "viscosity_unit",
        "eta_unit",
        "vis_unit",
    ],
    must=False,
)

print("识别到列:")
print("cation :", cat_col)
print("anion  :", ani_col)
print("T      :", t_col)
print("P      :", p_col)
print("vis    :", vis_col)
print("T_unit :", t_unit_col)
print("P_unit :", p_unit_col)
print("vis_unit:", vis_unit_col)

识别到列:
cation : cation_smiles
anion  : anion_smiles
T      : temperature, k
P      : pressure, kpa
vis    : Viscosity, Pa•s
T_unit : None
P_unit : None
vis_unit: None


In [4]:
# ============================================================
# Cell 3 | 保留原始列 + 基础重命名 + 单位信息统一落到 *_unit_raw
# 统一规则：
# 1) 如果原始表有单独单位列，就直接读取
# 2) 如果没有单独单位列，但单位写在列名里，就从列名里提取单位
# ============================================================

# 先检查 Cell 2 是否已经成功运行
required_vars = [
    "cat_col",
    "ani_col",
    "vis_col",
    "t_col",
    "p_col",
]

missing_vars = [
    variable
    for variable in required_vars
    if variable not in globals()
]

if missing_vars:
    raise RuntimeError(
        f"下面这些变量还没定义：{missing_vars}\n"
        f"请先重新运行 Cell 2，再运行 Cell 3。"
    )

import re
import numpy as np
import pandas as pd

df = df_raw.copy()

# -----------------------------
# 1) 保留原始关键列
# -----------------------------
df["cation_smiles_raw"] = (
    df[cat_col].astype(str)
)

df["anion_smiles_raw"] = (
    df[ani_col].astype(str)
)

df["vis_raw"] = (
    df[vis_col].astype(str)
)

# 原始温度、压强列统一改名
df = df.rename(
    columns={
        t_col: "T_raw",
        p_col: "P_raw",
    }
)


# -----------------------------
# 2) 从列名中提取单位
# -----------------------------
def extract_unit_from_colname(colname):
    """
    从列名中尽量提取单位字符串。

    适配示例：
      temperature, k      -> k
      pressure, kpa       -> kpa
      Viscosity, Pa・s    -> Pa・s
      Viscosity, Pa·s     -> Pa·s
      Viscosity(Pa·s)     -> Pa·s
    """
    if colname is None:
        return np.nan

    text = str(colname).strip()

    text = (
        text
        .replace("\u3000", " ")
        .strip()
    )

    text_normalized = re.sub(
        r"\s+",
        " ",
        text,
    )

    # 优先提取括号中的单位
    match = re.search(
        r"\(([^()]*)\)",
        text_normalized,
    )

    if match:
        unit = match.group(1).strip()

        return (
            unit
            if unit != ""
            else np.nan
        )

    # 提取英文逗号或中文逗号后的内容
    parts = re.split(
        r"[,，]",
        text_normalized,
    )

    if len(parts) >= 2:
        unit = parts[-1].strip()

        return (
            unit
            if unit != ""
            else np.nan
        )

    return np.nan


# -----------------------------
# 3) 温度单位列
# -----------------------------
if (
    "t_unit_col" in globals()
    and t_unit_col is not None
):
    # 有独立温度单位列，直接读取
    df["temperature_unit_raw"] = (
        df[t_unit_col].astype(str)
    )
else:
    # 没有独立单位列，从温度列名提取
    inferred_t_unit = (
        extract_unit_from_colname(t_col)
    )

    df["temperature_unit_raw"] = (
        inferred_t_unit
    )


# -----------------------------
# 4) 压强单位列
# -----------------------------
if (
    "p_unit_col" in globals()
    and p_unit_col is not None
):
    # 有独立压强单位列，直接读取
    df["pressure_unit_raw"] = (
        df[p_unit_col].astype(str)
    )
else:
    # 没有独立单位列，从压强列名提取
    inferred_p_unit = (
        extract_unit_from_colname(p_col)
    )

    df["pressure_unit_raw"] = (
        inferred_p_unit
    )


# -----------------------------
# 5) 粘度单位列
# -----------------------------
if (
    "vis_unit_col" in globals()
    and vis_unit_col is not None
):
    # 有独立粘度单位列，直接读取
    df["viscosity_unit_raw"] = (
        df[vis_unit_col].astype(str)
    )
else:
    # 没有独立单位列，从粘度列名提取
    inferred_vis_unit = (
        extract_unit_from_colname(vis_col)
    )

    df["viscosity_unit_raw"] = (
        inferred_vis_unit
    )


# -----------------------------
# 6) 预览
# -----------------------------
keep_preview = [
    "cation_smiles_raw",
    "anion_smiles_raw",
    "T_raw",
    "temperature_unit_raw",
    "P_raw",
    "pressure_unit_raw",
    "vis_raw",
    "viscosity_unit_raw",
]

print("Cell 3 done.")
print("shape =", df.shape)

display(
    df[keep_preview].head()
)

Cell 3 done.
shape = (24942, 12)


,cation_smiles_raw,anion_smiles_raw,T_raw,temperature_unit_raw,P_raw,pressure_unit_raw,vis_raw,viscosity_unit_raw
0,CC[S+](CC)CCOCCOCCOCC[S+](CC)CC,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,298.15,k,101.325,kpa,0.128 ± 0.013,Pa•s
1,CC[S+](CC)CCOCCOCCOCC[S+](CC)CC,N#C[N-]C#N,298.15,k,101.325,kpa,0.143 ± 0.016,Pa•s
2,CC[S+](CC)CCOCCOCC[S+](CC)CC,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,298.15,k,101.325,kpa,0.236 ± 0.038,Pa•s
3,CC[S+](CC)CCOCCOCC[S+](CC)CC,N#C[N-]C#N,298.15,k,101.325,kpa,0.361 ± 0.075,Pa•s
4,C[C@H]([N+](C)(C)C)[C@H](O)c1ccccc1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,296.25,k,100.000,kpa,0.482 ± 0.039,Pa•s


In [5]:
# ============================================================
# Cell 4 | SMILES 清洗 + RDKit 标准化
# ============================================================
def clean_smiles_text(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()
    s = s.replace(" ", "")
    s = s.replace("\u3000", "")

    return (
        s
        if s != ""
        else np.nan
    )


def canonicalize_smiles(s):
    if pd.isna(s):
        return np.nan, False

    try:
        mol = Chem.MolFromSmiles(s)

        if mol is None:
            return np.nan, False

        canonical = Chem.MolToSmiles(
            mol,
            canonical=True,
        )

        return canonical, True

    except Exception:
        return np.nan, False


df["cation_smiles_clean"] = (
    df["cation_smiles_raw"]
    .map(clean_smiles_text)
)

df["anion_smiles_clean"] = (
    df["anion_smiles_raw"]
    .map(clean_smiles_text)
)

cat_res = (
    df["cation_smiles_clean"]
    .map(canonicalize_smiles)
)

ani_res = (
    df["anion_smiles_clean"]
    .map(canonicalize_smiles)
)

df["cation_smiles"] = [
    result[0]
    for result in cat_res
]

df["cation_ok"] = [
    result[1]
    for result in cat_res
]

df["anion_smiles"] = [
    result[0]
    for result in ani_res
]

df["anion_ok"] = [
    result[1]
    for result in ani_res
]

bad_smiles = df[
    (~df["cation_ok"])
    | (~df["anion_ok"])
].copy()

good = df[
    df["cation_ok"]
    & df["anion_ok"]
].copy()

bad_path = (
    OUT_DIR
    / "bad_smiles_viscosity.csv"
)

bad_smiles.to_csv(
    bad_path,
    index=False,
    encoding="utf-8-sig",
)

print("good rows:", len(good))
print("bad smiles rows:", len(bad_smiles))
print("saved:", bad_path)

good rows: 24942
bad smiles rows: 0
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\bad_smiles_viscosity.csv


In [6]:
# ============================================================
# Cell 5 | 解析 viscosity 数值与误差
# 支持：
#   1.234 ± 0.005
#   1.234+/-0.005
#   1.234+-0.005
#   1.234
# ============================================================
import re


def parse_pm(x):
    if pd.isna(x):
        return np.nan, np.nan

    s = str(x).strip()

    s = (
        s
        .replace("+/-", "±")
        .replace("+-", "±")
    )

    parts = re.split(
        r"\s*±\s*",
        s,
    )

    try:
        if len(parts) == 2:
            return (
                float(parts[0]),
                float(parts[1]),
            )

        return (
            float(parts[0]),
            np.nan,
        )

    except Exception:
        return np.nan, np.nan


good[
    [
        "vis_mean_rawnum",
        "vis_sigma_rawnum",
    ]
] = good["vis_raw"].apply(
    lambda x: pd.Series(
        parse_pm(x)
    )
)

bad_parse = good[
    good["vis_mean_rawnum"].isna()
].copy()

ok = good[
    good["vis_mean_rawnum"].notna()
].copy()

bad_parse_path = (
    OUT_DIR
    / "bad_vis_parse.csv"
)

bad_parse.to_csv(
    bad_parse_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "vis 解析失败行数:",
    len(bad_parse),
)

print("saved:", bad_parse_path)

print(
    ok[
        [
            "vis_raw",
            "vis_mean_rawnum",
            "vis_sigma_rawnum",
        ]
    ].head()
)

vis 解析失败行数: 0
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\bad_vis_parse.csv
         vis_raw  vis_mean_rawnum  vis_sigma_rawnum
0  0.128 ± 0.013            0.128             0.013
1  0.143 ± 0.016            0.143             0.016
2  0.236 ± 0.038            0.236             0.038
3  0.361 ± 0.075            0.361             0.075
4  0.482 ± 0.039            0.482             0.039


In [7]:
# ============================================================
# Cell 6 | 温度、压强、粘度单位统一
# 目标：
#   T_K
#   P_bar
#   vis_mpas
#   vis_sigma_mpas
# 说明：
#   - 如果没有独立的 viscosity_unit_raw，但列名已写明单位，则 Cell 3 已提取
#   - 粘度统一到 mPa·s
# ============================================================
def normalize_temperature(v, unit):
    if pd.isna(v):
        return np.nan

    try:
        x = float(v)
    except Exception:
        return np.nan

    unit_text = (
        str(unit).strip().lower()
        if pd.notna(unit)
        else ""
    )

    if (
        "k" in unit_text
        and "c" not in unit_text
    ):
        return x

    if (
        "c" in unit_text
        or "℃" in unit_text
    ):
        return x + 273.15

    # 无法识别单位时：
    # 如果数值明显像K，就保留；否则按NaN
    if x > 170:
        return x

    return np.nan


def normalize_pressure_to_bar(v, unit):
    if pd.isna(v):
        return np.nan

    try:
        x = float(v)
    except Exception:
        return np.nan

    unit_text = (
        str(unit).strip().lower()
        if pd.notna(unit)
        else ""
    )

    if "bar" in unit_text:
        return x

    if "kpa" in unit_text:
        return x / 100.0

    if "mpa" in unit_text:
        return x * 10.0

    if (
        unit_text == ""
        or pd.isna(unit)
    ):
        # 没写就默认原始pressure是kPa
        return x / 100.0

    return np.nan


def normalize_viscosity_to_mpas(v, unit):
    if pd.isna(v):
        return np.nan

    try:
        x = float(v)
    except Exception:
        return np.nan

    unit_text = (
        str(unit)
        .strip()
        .lower()
        .replace("•", "·")
        if pd.notna(unit)
        else ""
    )

    unit_simple = (
        unit_text
        .replace(" ", "")
        .replace("·", "")
        .replace("*", "")
        .replace("/", "")
    )

    # mPa·s / cP
    if (
        ("mpa" in unit_text and "s" in unit_text)
        or ("mpas" in unit_simple)
    ):
        return x

    if (
        "cp" in unit_simple
        or "centipoise" in unit_simple
    ):
        return x

    # Pa·s -> mPa·s
    if (
        (
            "pa" in unit_text
            and "s" in unit_text
            and "mpa" not in unit_text
        )
        or ("pas" == unit_simple)
        or ("pasecond" in unit_simple)
    ):
        return x * 1000.0

    # 无单位时保守处理：返回原值
    if (
        unit_text == ""
        or pd.isna(unit)
    ):
        return x

    return np.nan


ok["T_K"] = ok.apply(
    lambda row: normalize_temperature(
        row["T_raw"],
        row["temperature_unit_raw"],
    ),
    axis=1,
)

ok["P_bar"] = ok.apply(
    lambda row: normalize_pressure_to_bar(
        row["P_raw"],
        row["pressure_unit_raw"],
    ),
    axis=1,
)

ok["vis_mpas"] = ok.apply(
    lambda row: normalize_viscosity_to_mpas(
        row["vis_mean_rawnum"],
        row["viscosity_unit_raw"],
    ),
    axis=1,
)

ok["vis_sigma_mpas"] = ok.apply(
    lambda row: normalize_viscosity_to_mpas(
        row["vis_sigma_rawnum"],
        row["viscosity_unit_raw"],
    ),
    axis=1,
)

print(
    ok[
        [
            "T_raw",
            "temperature_unit_raw",
            "T_K",
            "P_raw",
            "pressure_unit_raw",
            "P_bar",
            "vis_raw",
            "viscosity_unit_raw",
            "vis_mpas",
            "vis_sigma_mpas",
        ]
    ].head()
)

    T_raw temperature_unit_raw     T_K    P_raw pressure_unit_raw    P_bar  \
0  298.15                    k  298.15  101.325               kpa  1.01325   
1  298.15                    k  298.15  101.325               kpa  1.01325   
2  298.15                    k  298.15  101.325               kpa  1.01325   
3  298.15                    k  298.15  101.325               kpa  1.01325   
4  296.25                    k  296.25  100.000               kpa  1.00000   

         vis_raw viscosity_unit_raw  vis_mpas  vis_sigma_mpas  
0  0.128 ± 0.013               Pa•s     128.0            13.0  
1  0.143 ± 0.016               Pa•s     143.0            16.0  
2  0.236 ± 0.038               Pa•s     236.0            38.0  
3  0.361 ± 0.075               Pa•s     361.0            75.0  
4  0.482 ± 0.039               Pa•s     482.0            39.0  


In [8]:
# ============================================================
# Cell 7 | 基础质控过滤
# ============================================================
# 缺失
df_missing = ok[
    ok["T_K"].isna()
    | ok["P_bar"].isna()
    | ok["vis_mpas"].isna()
].copy()

missing_path = (
    OUT_DIR
    / "bad_missing_vis_TP.csv"
)

df_missing.to_csv(
    missing_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "missing rows saved to:",
    missing_path,
    "rows =",
    len(df_missing),
)

df = ok[
    ok["T_K"].notna()
    & ok["P_bar"].notna()
    & ok["vis_mpas"].notna()
].copy()

# 简单物理范围过滤（很宽松）
df_outlier = df[
    (df["vis_mpas"] <= 0)
    | (df["vis_mpas"] < 0.01)
    | (df["vis_mpas"] > 1e5)
].copy()

outlier_path = (
    OUT_DIR
    / "vis_suspicious_rows.csv"
)

df_outlier.to_csv(
    outlier_path,
    index=False,
    encoding="utf-8-sig",
)

print(
    "outlier rows saved to:",
    outlier_path,
    "rows =",
    len(df_outlier),
)

# 硬过滤：vis <= 0 的不要
df = df[
    df["vis_mpas"] > 0
].copy()

print(
    "after QC shape =",
    df.shape,
)

missing rows saved to: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\bad_missing_vis_TP.csv rows = 0
outlier rows saved to: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\vis_suspicious_rows.csv rows = 71
after QC shape = (24942, 22)


In [9]:
# ============================================================
# Cell 8 | 生成 IL_key + T/P 容差分组
# ============================================================
df["IL_key"] = (
    df["cation_smiles"]
    + "||"
    + df["anion_smiles"]
)

# 与 density 保持同类思路
T_tol = 0.5
P_tol = 0.05

df["T_group"] = (
    df["T_K"] / T_tol
).round() * T_tol

df["P_group"] = (
    df["P_bar"] / P_tol
).round() * P_tol

print(
    df[
        [
            "cation_smiles",
            "anion_smiles",
            "IL_key",
            "T_K",
            "T_group",
            "P_bar",
            "P_group",
            "vis_mpas",
            "vis_sigma_mpas",
        ]
    ].head()
)

                          cation_smiles  \
0       CC[S+](CC)CCOCCOCCOCC[S+](CC)CC   
1       CC[S+](CC)CCOCCOCCOCC[S+](CC)CC   
2          CC[S+](CC)CCOCCOCC[S+](CC)CC   
3          CC[S+](CC)CCOCCOCC[S+](CC)CC   
4  C[C@@H]([C@H](O)c1ccccc1)[N+](C)(C)C   

                             anion_smiles  \
0  O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F   
1                              N#C[N-]C#N   
2  O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F   
3                              N#C[N-]C#N   
4  O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F   

                                              IL_key     T_K  T_group  \
0  CC[S+](CC)CCOCCOCCOCC[S+](CC)CC||O=S(=O)([N-]S...  298.15    298.0   
1        CC[S+](CC)CCOCCOCCOCC[S+](CC)CC||N#C[N-]C#N  298.15    298.0   
2  CC[S+](CC)CCOCCOCC[S+](CC)CC||O=S(=O)([N-]S(=O...  298.15    298.0   
3           CC[S+](CC)CCOCCOCC[S+](CC)CC||N#C[N-]C#N  298.15    298.0   
4  C[C@@H]([C@H](O)c1ccccc1)[N+](C)(C)C||O=S(=O)(...  296.25    296.0   

     P_bar  P_group  vis_mpas  

In [10]:
# ============================================================
# Cell 9 | Step 1: 完全重复 / 同容差bin内重复整理
# 先去除完全重复行，再对同一 viscosity 值的重复记录保留最接近 group 的一条
# ============================================================
# 去完全重复
df1 = df.drop_duplicates().copy()

# 用数值字符串构造重复键
df1["vis_key"] = (
    df1["vis_mpas"]
    .round(8)
    .astype(str)
    + "_"
    + df1["vis_sigma_mpas"]
    .fillna(-999999)
    .round(8)
    .astype(str)
)

df1["T_dist"] = (
    df1["T_K"]
    - df1["T_group"]
).abs()

df1["P_dist"] = (
    df1["P_bar"]
    - df1["P_group"]
).abs()

df1["dist_sum"] = (
    df1["T_dist"]
    + df1["P_dist"]
)

group_cols_step1 = [
    "IL_key",
    "T_group",
    "P_group",
    "vis_key",
]

df_unique = (
    df1
    .sort_values(
        group_cols_step1
        + ["dist_sum"]
    )
    .groupby(
        group_cols_step1,
        as_index=False,
    )
    .head(1)
    .copy()
)

out_unique = (
    OUT_DIR
    / "viscosity_unique_tolbin.csv"
)

df_unique.to_csv(
    out_unique,
    index=False,
    encoding="utf-8-sig",
)

print("saved:", out_unique)
print("Step1 原始行数:", len(df))
print("Step1 去重后行数:", len(df_unique))

saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\viscosity_unique_tolbin.csv
Step1 原始行数: 24942
Step1 去重后行数: 24777


In [11]:
# ============================================================
# Cell 10 | Step 2: 分成“可加权”和“不可加权”两部分
# 可加权：sigma > 0
# 不可加权：sigma 缺失或 sigma <= 0
# ============================================================
df_w = df_unique[
    df_unique["vis_sigma_mpas"].notna()
    & (
        df_unique["vis_sigma_mpas"]
        > 0
    )
].copy()

df_nw = df_unique[
    df_unique["vis_sigma_mpas"].isna()
    | (
        df_unique["vis_sigma_mpas"]
        <= 0
    )
].copy()

print(
    "weighted rows    :",
    len(df_w),
)

print(
    "non-weighted rows:",
    len(df_nw),
)

weighted rows    : 24118
non-weighted rows: 659


In [12]:
# ============================================================
# Cell 11 | Step 3A: 对可加权部分做 1/sigma^2 加权平均
# ============================================================
df_w["w"] = (
    1.0
    / (
        df_w["vis_sigma_mpas"]
        ** 2
    )
)

group_cols = [
    "IL_key",
    "T_group",
    "P_group",
]


def weighted_summary(group):
    weights = (
        group["w"]
        .to_numpy()
    )

    viscosity = (
        group["vis_mpas"]
        .to_numpy()
    )

    sum_w = weights.sum()

    viscosity_weighted_mean = (
        (weights * viscosity).sum()
        / sum_w
    )

    viscosity_weighted_sigma = (
        np.sqrt(
            1.0 / sum_w
        )
    )

    return pd.Series(
        {
            "vis_rep_mpas": (
                viscosity_weighted_mean
            ),
            "vis_sigma_rep_mpas": (
                viscosity_weighted_sigma
            ),
            "n_points": len(group),
            "sum_w": sum_w,
            "T_rep": float(
                np.nanmedian(
                    group["T_K"]
                )
            ),
            "P_rep": float(
                np.nanmedian(
                    group["P_bar"]
                )
            ),
            "agg_method": (
                "weighted_by_1_over_sigma2"
            ),
        }
    )


df_w_agg = (
    df_w
    .groupby(
        group_cols,
        as_index=False,
    )
    .apply(weighted_summary)
    .reset_index(drop=True)
)

print(df_w_agg.head())

print(
    "weighted aggregated rows =",
    len(df_w_agg),
)

                                              IL_key  T_group  P_group  \
0  C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...    303.0      1.0   
1  C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...    313.0      1.0   
2  C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...    323.0      1.0   
3  C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...    333.0      1.0   
4  C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...    343.0      1.0   

   vis_rep_mpas  vis_sigma_rep_mpas  n_points     sum_w  T_rep    P_rep  \
0         62.60                2.60         1  0.147929  303.0  1.01325   
1         40.40                1.70         1  0.346021  313.0  1.01325   
2         28.00                1.20         1  0.694444  323.0  1.01325   
3         20.47                0.86         1  1.352082  333.0  1.01325   
4         15.47                0.65         1  2.366864  343.0  1.01325   

                  agg_method  
0  weighted_by_1_over_sigma2  
1  weighted_by_1_over_sigma2  
2  weighted

C:\Users\86178\AppData\Local\Temp\ipykernel_12732\3372454833.py:71: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_w


In [13]:
# ============================================================
# Cell 12 | Step 3B: 对不可加权部分做普通平均
# 这是为了避免把没 sigma 的数据全扔掉
# ============================================================
def simple_summary(group):
    viscosity = (
        group["vis_mpas"]
        .to_numpy()
    )

    if len(viscosity) == 1:
        sigma_rep = np.nan
    else:
        sigma_rep = float(
            np.std(
                viscosity,
                ddof=1,
            )
        )

    return pd.Series(
        {
            "vis_rep_mpas": float(
                np.mean(viscosity)
            ),
            "vis_sigma_rep_mpas": (
                sigma_rep
            ),
            "n_points": len(group),
            "sum_w": np.nan,
            "T_rep": float(
                np.nanmedian(
                    group["T_K"]
                )
            ),
            "P_rep": float(
                np.nanmedian(
                    group["P_bar"]
                )
            ),
            "agg_method": (
                "simple_mean_no_valid_sigma"
            ),
        }
    )


if len(df_nw) > 0:
    df_nw_agg = (
        df_nw
        .groupby(
            group_cols,
            as_index=False,
        )
        .apply(simple_summary)
        .reset_index(drop=True)
    )
else:
    df_nw_agg = pd.DataFrame(
        columns=[
            "IL_key",
            "T_group",
            "P_group",
            "vis_rep_mpas",
            "vis_sigma_rep_mpas",
            "n_points",
            "sum_w",
            "T_rep",
            "P_rep",
            "agg_method",
        ]
    )

print(df_nw_agg.head())

print(
    "non-weighted aggregated rows =",
    len(df_nw_agg),
)

                                              IL_key  T_group  P_group  \
0  C=CC[N+](C)(C)CCn1cc[n+](C)c1||N#C[N-]C#N.N#C[...    298.5      1.0   
1  C=CC[N+](C)(C)CCn1cc[n+](C)c1||N#C[N-]C#N.N#C[...    301.0      1.0   
2  C=CC[N+](C)(C)CCn1cc[n+](C)c1||N#C[N-]C#N.N#C[...    303.5      1.0   
3  C=CC[N+](C)(C)CCn1cc[n+](C)c1||N#C[N-]C#N.N#C[...    306.5      1.0   
4  C=CC[N+](C)(C)CCn1cc[n+](C)c1||N#C[N-]C#N.N#C[...    310.5      1.0   

   vis_rep_mpas  vis_sigma_rep_mpas  n_points  sum_w   T_rep    P_rep  \
0         569.2                 NaN         1    NaN  298.35  1.01325   
1         473.0                 NaN         1    NaN  300.75  1.01325   
2         389.3                 NaN         1    NaN  303.35  1.01325   
3         307.1                 NaN         1    NaN  306.35  1.01325   
4         230.3                 NaN         1    NaN  310.35  1.01325   

                   agg_method  
0  simple_mean_no_valid_sigma  
1  simple_mean_no_valid_sigma  
2  simple_mean_no_va

C:\Users\86178\AppData\Local\Temp\ipykernel_12732\2502959970.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_nw


In [14]:
# ============================================================
# Cell 13 | 合并聚合结果 + 补回 smiles + 补充 lg/误差/样本权重
# 说明：
# 1) 让 viscosity_weighted_tolbin.csv 直接成为粘度进入 ml_ready_build 的统一标准输入
# 2) 当前法A先用主值 lg_eta
# 3) 同时提前保留法B需要的信息：lg_eta_sigma / sample_weight_from_sigma
# 4) 这里的 lg_eta 本质上就是 log10(eta)，论文中可写作 lg η
# ============================================================
df_agg = pd.concat(
    [
        df_w_agg,
        df_nw_agg,
    ],
    axis=0,
    ignore_index=True,
)

il_map = (
    df_unique[
        [
            "IL_key",
            "cation_smiles",
            "anion_smiles",
            "cation_smiles_raw",
            "anion_smiles_raw",
        ]
    ]
    .drop_duplicates("IL_key")
)

df_agg = df_agg.merge(
    il_map,
    on="IL_key",
    how="left",
)

# -----------------------------
# 1) 主值取常用对数
#    lg_eta = lg(eta) = log10(eta)
# -----------------------------
df_agg["lg_eta"] = np.log10(
    df_agg["vis_rep_mpas"]
)

# -----------------------------
# 2) 误差传播到lg空间
#    sigma_lg(y) ≈ sigma_y / (y * ln(10))
# -----------------------------
df_agg["lg_eta_sigma"] = np.nan

valid_sigma_mask = (
    df_agg[
        "vis_sigma_rep_mpas"
    ].notna()
    & (
        df_agg[
            "vis_sigma_rep_mpas"
        ] > 0
    )
    & df_agg[
        "vis_rep_mpas"
    ].notna()
    & (
        df_agg[
            "vis_rep_mpas"
        ] > 0
    )
)

df_agg.loc[
    valid_sigma_mask,
    "lg_eta_sigma",
] = (
    df_agg.loc[
        valid_sigma_mask,
        "vis_sigma_rep_mpas",
    ]
    /
    (
        df_agg.loc[
            valid_sigma_mask,
            "vis_rep_mpas",
        ]
        * np.log(10)
    )
)

# -----------------------------
# 3) 为后期法B提前准备样本权重
#    w = 1 / sigma_lg^2
# -----------------------------
df_agg[
    "sample_weight_from_sigma"
] = np.nan

valid_weight_mask = (
    df_agg[
        "lg_eta_sigma"
    ].notna()
    & (
        df_agg[
            "lg_eta_sigma"
        ] > 0
    )
)

df_agg.loc[
    valid_weight_mask,
    "sample_weight_from_sigma",
] = (
    1.0
    /
    (
        df_agg.loc[
            valid_weight_mask,
            "lg_eta_sigma",
        ]
        ** 2
    )
)

# -----------------------------
# 4) 保存weighted_tolbin
# -----------------------------
out_weighted = (
    OUT_DIR
    / "viscosity_weighted_tolbin.csv"
)

df_agg.to_csv(
    out_weighted,
    index=False,
    encoding="utf-8-sig",
)

print("saved:", out_weighted)
print("rows :", len(df_agg))

display(df_agg.head())

saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\viscosity_weighted_tolbin.csv
rows : 16216


,IL_key,T_group,P_group,vis_rep_mpas,vis_sigma_rep_mpas,n_points,sum_w,T_rep,P_rep,agg_method,cation_smiles,anion_smiles,cation_smiles_raw,anion_smiles_raw,lg_eta,lg_eta_sigma,sample_weight_from_sigma
0,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,303.0,1.0,62.60,2.60,1,0.147929,303.0,1.01325,weighted_by_1_over_sigma2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,1.796574,0.018038,3073.500923
1,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,313.0,1.0,40.40,1.70,1,0.346021,313.0,1.01325,weighted_by_1_over_sigma2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,1.606381,0.018275,2994.306581
2,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,323.0,1.0,28.00,1.20,1,0.694444,323.0,1.01325,weighted_by_1_over_sigma2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,1.447158,0.018613,2886.588971
3,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,333.0,1.0,20.47,0.86,1,1.352082,333.0,1.01325,weighted_by_1_over_sigma2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,1.311118,0.018246,3003.794102
4,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,343.0,1.0,15.47,0.65,1,2.366864,343.0,1.01325,weighted_by_1_over_sigma2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,1.189490,0.018248,3003.207166


In [15]:
# ============================================================
# Cell 14 | 最终 ml-ready 输出
# 说明：
# 1) Cell 13 已经完成：
#    - lg_eta
#    - lg_eta_sigma
#    - sample_weight_from_sigma
# 2) 这里不再重复计算，只做最终列整理与导出
# 3) 文件名沿用旧命名，直接覆盖旧文件
# ============================================================
df_ml = df_agg.copy()

# -----------------------------
# 1) 最终列
# -----------------------------
final_cols = [
    "cation_smiles",
    "anion_smiles",
    "IL_key",
    "T_group",
    "P_group",
    "T_rep",
    "P_rep",
    "vis_rep_mpas",
    "vis_sigma_rep_mpas",
    "lg_eta",
    "lg_eta_sigma",
    "sample_weight_from_sigma",
    "n_points",
    "agg_method",
]

df_ml = df_ml[
    final_cols
].copy()

# -----------------------------
# 2) 排序，方便后续对齐
# -----------------------------
df_ml = (
    df_ml
    .sort_values(
        by=[
            "cation_smiles",
            "anion_smiles",
            "T_rep",
            "P_rep",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# -----------------------------
# 3) 保存
# -----------------------------
out_ml = (
    OUT_DIR
    / "vis_ml_ready.csv"
)

df_ml.to_csv(
    out_ml,
    index=False,
    encoding="utf-8-sig",
)

print("saved:", out_ml)
print("ml-ready rows:", len(df_ml))

display(df_ml.head())

saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim\vis_ml_ready.csv
ml-ready rows: 16216


,cation_smiles,anion_smiles,IL_key,T_group,P_group,T_rep,P_rep,vis_rep_mpas,vis_sigma_rep_mpas,lg_eta,lg_eta_sigma,sample_weight_from_sigma,n_points,agg_method
0,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,303.0,1.0,303.0,1.01325,62.60,2.60,1.796574,0.018038,3073.500923,1,weighted_by_1_over_sigma2
1,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,313.0,1.0,313.0,1.01325,40.40,1.70,1.606381,0.018275,2994.306581,1,weighted_by_1_over_sigma2
2,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,323.0,1.0,323.0,1.01325,28.00,1.20,1.447158,0.018613,2886.588971,1,weighted_by_1_over_sigma2
3,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,333.0,1.0,333.0,1.01325,20.47,0.86,1.311118,0.018246,3003.794102,1,weighted_by_1_over_sigma2
4,C1CC[NH2+]C1,O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(F)(F)F,C1CC[NH2+]C1||O=S(=O)([N-]S(=O)(=O)C(F)(F)F)C(...,343.0,1.0,343.0,1.01325,15.47,0.65,1.189490,0.018248,3003.207166,1,weighted_by_1_over_sigma2


In [16]:
# ============================================================
# Cell 15 | 输出目录检查
# ============================================================
print("OUT_DIR =", OUT_DIR)

for path in sorted(
    OUT_DIR.iterdir()
):
    print(path.name)

OUT_DIR = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\data\viscosity\interim
bad_missing_vis_TP.csv
bad_smiles_viscosity.csv
bad_vis_parse.csv
vis_ml_ready.csv
vis_suspicious_rows.csv
viscosity_unique_tolbin.csv
viscosity_weighted_tolbin.csv
